# Part 2 · Loading a saved model and writing an inference function

`predict.py` loads the model once and turns five raw values into a probability. It does not import Gradio.

In [1]:
from pathlib import Path

import churn_model

if not Path("churn_model.joblib").exists():
    churn_model.main()  # trains in THIS environment and saves the model
else:
    print("churn_model.joblib already exists")

churn_model.joblib already exists


In [2]:
from predict import predict_churn

print(predict_churn(3, 95.0, 4, "month-to-month", "cheque"))
print(round(predict_churn(60, 40.0, 0, "two-year", "card"), 3))

0.9286141836018518
0.041


## A missing monthly charge
The pipeline's imputer fills it with the training median, because the model was trained on data with gaps.

In [3]:
print(predict_churn(3, None, 4, "month-to-month", "cheque"))

0.897595025691833


## The columns must match
The preprocessing step selects columns by name, so a wrong name stops the pipeline.

In [4]:
import pandas as pd

from predict import model

row = pd.DataFrame(
    [
        {
            "tenure_months": 3,
            "monthly_charge": 95.0,
            "support_calls": 4,
            "contract": "month-to-month",
            "payment": "cheque",
        }
    ]
)
bad = row.rename(columns={"tenure_months": "tenure"})
try:
    model.predict_proba(bad)
except ValueError as error:
    print("ValueError:", error)

ValueError: columns are missing: {'tenure_months'}


## Test it without a web page

In [5]:
import subprocess
import sys

result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "--color=no"], capture_output=True, text=True
)
print(result.stdout)

..                                                                       [100%]
2 passed in 1.39s



**Try it:** time `joblib.load` against 100 calls to `predict_churn`, as in Part 1, and add a third test of your own.